# Preprocessing Teks

In [15]:
%pip install pandas nltk Sastrawi polars

Note: you may need to restart the kernel to use updated packages.


## Library

In [16]:
import re
import html
import polars as pl
import nltk
# import spacy
import unicodedata
from functools import lru_cache
from collections import Counter

# import pandas as pd

from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [17]:
nltk.download("stopwords")

# Model spaCy untuk bahasa Indonesia
# spacy.cli.download("xx_ent_wiki_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Musa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Helper Fungsion

In [18]:
def to_list(v):
    """map_elements memberi pl.Series untuk kolom list -> ubah ke list Python."""
    if v is None: return []
    if isinstance(v, pl.Series): return v.to_list()
    return list(v)
 
def _apply_str(df, col, out, fn):
    return df.with_columns(pl.col(col).map_elements(fn, return_dtype=pl.Utf8).alias(out))
 
def _apply_list(df, col, out, fn):
    return df.with_columns(
        pl.col(col).map_elements(lambda v: fn(to_list(v)), return_dtype=pl.List(pl.Utf8)).alias(out))

## Dataset

In [19]:
df = pl.read_csv("data_twiter/timnas_final.csv")

In [20]:
df

type,id,url,twitterUrl,text,source,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,lang,bookmarkCount,isReply,inReplyToId,conversationId,inReplyToUserId,inReplyToUsername,isPinned,author,extendedEntities,card,place,entities,reply_to_user_results,quoted_tweet_results,quoted_tweet,retweeted_tweet,isConversationControlled,searchTermIndex,isQuote
str,i64,str,str,str,str,i64,i64,i64,i64,i64,str,str,i64,bool,i64,i64,i64,str,bool,str,str,str,str,str,str,str,str,str,bool,i64,bool
"""tweet""",2108135985704001818,"""https://x.com/indozonemedia/st…","""https://twitter.com/indozoneme…","""Elkan Baggott mengaku bangga j…",null,0,0,0,0,45,"""Thu Oct 08 10:02:40 +0000 2026""","""in""",0,false,null,2108135985704001818,null,null,false,"""{'type': 'user', 'userName': '…","""{}""","""{'binding_values': [{'key': 'p…","""{}""","""{'urls': [{'display_url': 'are…",null,null,null,null,false,0,null
"""tweet""",2108133007307702498,"""https://x.com/jawapos/status/2…","""https://twitter.com/jawapos/st…","""Pembalap MotoGP Fermin Aldegue…",null,0,0,0,0,162,"""Thu Oct 08 09:50:50 +0000 2026""","""in""",0,false,null,2108133007307702498,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'additional_media_…",null,"""{}""","""{'hashtags': [], 'symbols': []…",null,null,null,null,false,0,null
"""tweet""",2108132172431540703,"""https://x.com/karedd_/status/2…","""https://twitter.com/karedd_/st…","""ngeliat banyaknya reaksi ""ajai…",null,0,0,0,0,2,"""Thu Oct 08 09:47:31 +0000 2026""","""in""",0,false,null,2108132172431540703,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{}""",null,null,null,null,false,0,null
"""tweet""",2108128642283024777,"""https://x.com/milyarderkalcer/…","""https://twitter.com/milyarderk…","""Terpantau hanya akun ini yang …",null,0,0,0,0,10,"""Thu Oct 08 09:33:30 +0000 2026""","""in""",0,false,null,2108128642283024777,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'display_url': 'pi…",null,"""{}""","""{}""",null,null,null,null,false,0,null
"""tweet""",2108128304180195697,"""https://x.com/sportskompascom/…","""https://twitter.com/sportskomp…","""Baca di sini: https://t.co/fED…",null,0,0,0,0,123,"""Thu Oct 08 09:32:09 +0000 2026""","""in""",0,false,null,2108128304180195697,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'display_url': 'pi…",null,"""{}""","""{'hashtags': [{'indices': [355…",null,null,null,null,false,0,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tweet""",2107361261055021384,"""https://x.com/ssbmasmas/status…","""https://twitter.com/ssbmasmas/…","""Apa pun Analisisnya, Mari Raya…",null,2,2,13,1,951,"""Tue Oct 06 06:44:12 +0000 2026""","""in""",0,false,null,2107361261055021384,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'display_url': 'pi…",null,"""{}""","""{'hashtags': [], 'symbols': []…",null,null,null,null,false,3,null
"""tweet""",2107360723806601308,"""https://x.com/bimo_mulyoto/sta…","""https://twitter.com/bimo_mulyo…","""Pantesan dukun Thailand nya ka…",null,0,0,0,0,14,"""Tue Oct 06 06:42:04 +0000 2026""","""in""",0,false,null,2107360723806601308,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{}""",null,"""{'rest_id': '21072002206313718…","""{'type': 'tweet', 'id': '21072…",null,false,3,true
"""tweet""",2107360659927355712,"""https://x.com/urekzi/status/21…","""https://twitter.com/urekzi/sta…","""Setuju sih ini, thailand kalah…",null,0,0,0,0,21,"""Tue Oct 06 06:41:48 +0000 2026""","""in""",0,false,null,2107360659927355712,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{}""",null,"""{'rest_id': '21072781472092858…","""{'type': 'tweet', 'id': '21072…",null,false,3,true


In [21]:
df = df.select("text")
df

text
str
"""Elkan Baggott mengaku bangga j…"
"""Pembalap MotoGP Fermin Aldegue…"
"""ngeliat banyaknya reaksi ""ajai…"
"""Terpantau hanya akun ini yang …"
"""Baca di sini: https://t.co/fED…"
…
"""Apa pun Analisisnya, Mari Raya…"
"""Pantesan dukun Thailand nya ka…"
"""Setuju sih ini, thailand kalah…"


## Noise Removal

Noise Removal digunakan untuk membersihkan atau menghapus elemen yang biasanya tidak dibutuhkan dalam analisis teks, seperti HTML, URL, alamat email, mention, karakter kontrol, dan spasi berlebih yang tidak relevan dan tidak memiliki nilai informasi untuk analisis teks.

Contoh yang dibersihkan: Tanda baca, emoji, tag HTML, URL, karakter khusus (seperti @, #, $), dan spasi berlebih (whitespace).

### Code

In [22]:
EMOJI = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF\uFE0F\u200d]")

In [23]:
def noise_removal(df, text_col, output_col="text_clean"):
    def clean(text):
        if text is None: return ""
        text = html.unescape(str(text))
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+|\b\w+\.(?:co|com|id)/\S+", " ", text)
        text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)
        text = re.sub(r"(?<!\w)@\w+", " ", text)
        text = re.sub(r"^\s*RT\b", " ", text)
        # hashtag: buang '#', pisahkan CamelCase
        text = re.sub(r"#(\w+)", lambda m: " " + re.sub(r"(?<=[a-z])(?=[A-Z])", " ", m.group(1)) + " ", text)
        text = EMOJI.sub(" ", text)
        text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
        return re.sub(r"\s+", " ", text).strip()
    return _apply_str(df, text_col, output_col, clean)
 
ROMAN = re.compile(r"^m{0,3}(cm|cd|d?c{0,3})(xc|xl|l?x{0,3})(ix|iv|v?i{0,3})$")
def _collapse(m):
    w = m.group(0)
    if ROMAN.match(w) or w == "www":      # III, XXX, www jangan diubah
        return w
    return re.sub(r"(\w)\1{2,}", r"\1", w)

In [24]:
df_noise = noise_removal(
    df, text_col="text"
)

In [25]:
df_noise

text,text_clean
str,str
"""Elkan Baggott mengaku bangga j…","""Elkan Baggott mengaku bangga j…"
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…"
…,…
"""Apa pun Analisisnya, Mari Raya…","""Apa pun Analisisnya, Mari Raya…"
"""Pantesan dukun Thailand nya ka…","""Pantesan dukun Thailand nya ka…"
"""Setuju sih ini, thailand kalah…","""Setuju sih ini, thailand kalah…"


In [26]:
print(type(df))


<class 'polars.dataframe.frame.DataFrame'>


## Normalisasi Teks

Normalisasi teks adalah proses menyeragamkan format teks, terutama huruf kapital, Unicode, dan spasi agar memiliki struktur linguistik yang seragam, terutama untuk teks tidak formal, bahasa gaul, singkatan, atau typo (typhographical errors).

Contoh: Mengubah kata "bgt" menjadi "banget", "udh" menjadi "sudah", atau menstandarisasi penulisan angka dan simbol mata uang.

### Code

In [27]:
SLANG = {
    "yg": "yang", "dg": "dengan", "dgn": "dengan", "utk": "untuk", "krn": "karena",
    "karna": "karena", "tdk": "tidak", "gak": "tidak", "ga": "tidak", "nggak": "tidak",
    "gk": "tidak", "enggak": "tidak", "ngga": "tidak", "bkn": "bukan", "bkan": "bukan",
    "sdh": "sudah", "udh": "sudah", "udah": "sudah", "blm": "belum", "jg": "juga",
    "jga": "juga", "sm": "sama", "sma": "sama", "klo": "kalau", "kalo": "kalau",
    "klu": "kalau", "trus": "terus", "tp": "tapi", "tpi": "tapi", "aja": "saja",
    "bgt": "banget", "emg": "memang", "emang": "memang", "org": "orang", "ornag": "orang",
    "gw": "saya", "gue": "saya", "gua": "saya", "lu": "kamu", "loe": "kamu",
    "dr": "dari", "dlm": "dalam", "pd": "pada", "sbg": "sebagai", "spt": "seperti",
    "mknya": "makanya", "pkiran": "pikiran", "stlah": "setelah", "mnjbat": "menjabat",
    "ko": "kok", "mentri": "menteri", "thn": "tahun", "nglantik": "melantik",
    "latarbelakang": "latar belakang", "ehh": "eh", "wajarlah": "wajar lah",
    "rg": "rocky gerung"
}

In [28]:
def text_normalization(df, text_col, output_col="text_normalized", slang=SLANG):
    def norm(text):
        if not text: return ""
        text = unicodedata.normalize("NFKC", text).lower()
        text = re.sub(r"[“”„‟]", '"', text)
        text = re.sub(r"[‘’‚‛]", "'", text)
        text = re.sub(r"[—–]", " - ", text)
        text = re.sub(r"\b\w*(\w)\1{2,}\w*\b", _collapse, text)       # mantaaap -> mantap (angka Romawi dikecualikan)
        text = re.sub(r"\brp\s?(\d)", r"rp \1", text)                # rp76 -> rp 76
        text = re.sub(r"\b([a-z]+)2\b", r"\1-\1", text)             # tiba2 -> tiba-tiba
        text = re.sub(r"\b[a-z0-9]+\b", lambda m: slang.get(m.group(0), m.group(0)), text)
        return re.sub(r"\s+", " ", text).strip()
    return _apply_str(df, text_col, output_col, norm)

In [29]:
df_normalized = text_normalization(
    df_noise, text_col="text_clean"
)

In [30]:
df_normalized

text,text_clean,text_normalized
str,str,str
"""Elkan Baggott mengaku bangga j…","""Elkan Baggott mengaku bangga j…","""elkan baggott mengaku bangga j…"
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…"
…,…,…
"""Apa pun Analisisnya, Mari Raya…","""Apa pun Analisisnya, Mari Raya…","""apa pun analisisnya, mari raya…"
"""Pantesan dukun Thailand nya ka…","""Pantesan dukun Thailand nya ka…","""pantesan dukun thailand nya ka…"
"""Setuju sih ini, thailand kalah…","""Setuju sih ini, thailand kalah…","""setuju sih ini, thailand kalah…"


## Tokenisasi

Memecah kalimat atau paragraf yang panjang menjadi unit-unit kecil yang disebut token (biasanya berupa kata atau frasa), hasilnya berupa list, bukan string..

Contoh: Kalimat "Saya sedang belajar NLP." dipecah menjadi: ["Saya", "sedang", "belajar", "NLP"].

### Code

In [31]:
# def tokenization(df, text_col, output_col="tokens"):
#     """
#     Memecah teks menjadi token kata.
#     Output setiap baris berupa list.
#     """
#     result = df.clone()

#     def tokenize_text(text):
#         if text is None:
#             return ""
#         # if pl.isna(text):
#         #     return []

#         text = str(text)

#         # Ambil rangkaian karakter alfanumerik.
#         # Kata dengan apostrof atau tanda hubung dipisah.
#         return re.findall(r"\b\w+\b", text, flags=re.UNICODE)

#     return df.with_columns(
#         df[text_col].map_elements(tokenize_text).alias(output_col)
#     )
#     # result[output_col] = result[text_col].apply(tokenize_text)
#     # return result

In [32]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")
def tokenization(df, text_col, output_col="tokens"):
    return df.with_columns(
        pl.col(text_col).map_elements(lambda t: TOKEN_RE.findall(t or ""),
                                      return_dtype=pl.List(pl.Utf8)).alias(output_col))
 

In [33]:
df_tokens = tokenization(
    df_normalized, text_col="text_normalized"
)

In [34]:
df_tokens

text,text_clean,text_normalized,tokens
str,str,str,list[str]
"""Elkan Baggott mengaku bangga j…","""Elkan Baggott mengaku bangga j…","""elkan baggott mengaku bangga j…","[""elkan"", ""baggott"", … ""2026""]"
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]"
…,…,…,…
"""Apa pun Analisisnya, Mari Raya…","""Apa pun Analisisnya, Mari Raya…","""apa pun analisisnya, mari raya…","[""apa"", ""pun"", … ""kemudian""]"
"""Pantesan dukun Thailand nya ka…","""Pantesan dukun Thailand nya ka…","""pantesan dukun thailand nya ka…","[""pantesan"", ""dukun"", … ""kalah""]"
"""Setuju sih ini, thailand kalah…","""Setuju sih ini, thailand kalah…","""setuju sih ini, thailand kalah…","[""setuju"", ""sih"", … ""semalem""]"


## Stopwords Removal

Menghapus kata-kata umum yang sering muncul dalam suatu bahasa namun memiliki sedikit atau bahkan tidak memiliki makna analitik yang kuat.

Contoh: dan, yang, di, dari, ke, untuk

### Code

In [35]:
# Stopword bahasa Indonesia
stop_words = set(stopwords.words("indonesian"))

# Tambahan stopword
custom_stopwords = {
    "yg", "dg", "dgn", "nya", "sih", "dong",
    "nih", "deh", "aja", "udah", "gak", "nggak"
}

stop_words.update(custom_stopwords)

In [36]:
def build_stopwords(keep_negation=True):
    sw = set(stopwords.words("indonesian"))
    sw |= {"sih", "dong", "nih", "deh", "kok", "lah", "kan", "nya", "tuh", "nah", "yah", "ya"}
    if keep_negation:
        sw -= {"tidak", "bukan", "tak", "belum", "jangan", "tanpa"}
    return sw
 
def stopword_removal(df, text_col, output_col="tokens_no_stopwords", stop_words=None):
    stop_words = stop_words or build_stopwords()
    return _apply_list(df, text_col, output_col,
                       lambda toks: [t for t in toks if t not in stop_words and len(t) > 1])

In [37]:
# stop_words

In [38]:
df_no_stopwords = stopword_removal(
    df_tokens, text_col="tokens"
)

In [39]:
df_no_stopwords

text,text_clean,text_normalized,tokens,tokens_no_stopwords
str,str,str,list[str],list[str]
"""Elkan Baggott mengaku bangga j…","""Elkan Baggott mengaku bangga j…","""elkan baggott mengaku bangga j…","[""elkan"", ""baggott"", … ""2026""]","[""elkan"", ""baggott"", … ""2026""]"
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]","[""ngeliat"", ""banyaknya"", … ""negara""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]","[""terpantau"", ""akun"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]"
…,…,…,…,…
"""Apa pun Analisisnya, Mari Raya…","""Apa pun Analisisnya, Mari Raya…","""apa pun analisisnya, mari raya…","[""apa"", ""pun"", … ""kemudian""]","[""analisisnya"", ""mari"", … ""lakukan""]"
"""Pantesan dukun Thailand nya ka…","""Pantesan dukun Thailand nya ka…","""pantesan dukun thailand nya ka…","[""pantesan"", ""dukun"", … ""kalah""]","[""pantesan"", ""dukun"", … ""kalah""]"
"""Setuju sih ini, thailand kalah…","""Setuju sih ini, thailand kalah…","""setuju sih ini, thailand kalah…","[""setuju"", ""sih"", … ""semalem""]","[""setuju"", ""thailand"", … ""semalem""]"


In [40]:
df_no_stopwords['tokens_no_stopwords'][6]

""
str
"""baca"""
"""calvin"""
"""verdonk"""
"""tak"""
"""merayakan"""
…
"""calvin"""
"""verdonk"""
"""timnas"""


## Stemming

Proses memotong imbuhan (prefiks, sufiks, infiks) pada sebuah kata untuk dikembalikan ke bentuk dasarnya (stem), tanpa selalu memperhatikan kaidah kamus yang valid.

Contoh: Kata "memakan", "dimakan", dan "termakan" akan diubah menjadi akar kata dasar "makan".

### Code

In [41]:
# Inisialisasi stemmer bahasa Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()


In [42]:
import polars as pl

def to_list(v):
    """Mengubah pl.Series atau iterables lainnya menjadi list Python."""
    if v is None: 
        return []
    if isinstance(v, pl.Series): 
        return v.to_list()
    if isinstance(v, list):
        return v
    return list(v)

def stemming(df: pl.DataFrame, text_col: str, output_col: str = "tokens_stemmed"):
    """
    Melakukan stemming menggunakan Sastrawi dengan aman dari noise metadata.
    """
    def stem_text(value):
        # Konversi nilai baris (bisa string, list, atau pl.Series) menjadi list token yang bersih
        if value is None:
            tokens = []
        elif isinstance(value, pl.Series):
            tokens = value.to_list()
        elif isinstance(value, list):
            tokens = value
        else:
            # Jika berupa string teks biasa (belum berupa list token)
            tokens = str(value).split()
            
        # Lakukan stemming pada setiap token
        return [stemmer.stem(str(token)) for token in tokens if token is not None]

    return df.with_columns(
        df[text_col].map_elements(
            stem_text,
            return_dtype=pl.List(pl.Utf8)
        ).alias(output_col)
    )

In [43]:
df_stemmed = stemming(
    df_no_stopwords, text_col="tokens_no_stopwords"
)

In [44]:
df_stemmed

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed
str,str,str,list[str],list[str],list[str]
"""Elkan Baggott mengaku bangga j…","""Elkan Baggott mengaku bangga j…","""elkan baggott mengaku bangga j…","[""elkan"", ""baggott"", … ""2026""]","[""elkan"", ""baggott"", … ""2026""]","[""elkan"", ""baggott"", … ""2026""]"
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]","[""balap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]","[""ngeliat"", ""banyaknya"", … ""negara""]","[""ngeliat"", ""banyak"", … ""negara""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]","[""terpantau"", ""akun"", … ""si""]","[""pantau"", ""akun"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]"
…,…,…,…,…,…
"""Apa pun Analisisnya, Mari Raya…","""Apa pun Analisisnya, Mari Raya…","""apa pun analisisnya, mari raya…","[""apa"", ""pun"", … ""kemudian""]","[""analisisnya"", ""mari"", … ""lakukan""]","[""analisis"", ""mari"", … ""laku""]"
"""Pantesan dukun Thailand nya ka…","""Pantesan dukun Thailand nya ka…","""pantesan dukun thailand nya ka…","[""pantesan"", ""dukun"", … ""kalah""]","[""pantesan"", ""dukun"", … ""kalah""]","[""pantesan"", ""dukun"", … ""kalah""]"
"""Setuju sih ini, thailand kalah…","""Setuju sih ini, thailand kalah…","""setuju sih ini, thailand kalah…","[""setuju"", ""sih"", … ""semalem""]","[""setuju"", ""thailand"", … ""semalem""]","[""tuju"", ""thailand"", … ""semalem""]"


In [45]:
# STEM_EXCEPTIONS = {"belasan": "belas"}
# PROTECTED = {"wantimpres", "gibran", "prabowo", "rocky", "gerung", "pratikno", "dpr"}

In [46]:
# _stemmer = StemmerFactory().create_stemmer()
# @lru_cache(maxsize=None)
# def _stem(tok):
#     if tok in PROTECTED or tok.isdigit() or "-" in tok: return tok
#     if tok in STEM_EXCEPTIONS: return STEM_EXCEPTIONS[tok]
#     return _stemmer.stem(tok)
    
# def stemming(df, text_col, output_col="tokens_stemmed"):
#     return _apply_list(df, text_col, output_col, lambda toks: [_stem(t) for t in toks])

In [47]:
df_stemmed['tokens_stemmed'][0]


""
str
"""elkan"""
"""baggott"""
"""aku"""
"""bangga"""
"""orang"""
…
"""juara"""
"""fifa"""
"""asean"""


## Lemmatization

Proses mengubah kata berimbuhan menjadi bentuk dasarnya berdasarkan kamus dan tata bahasa yang valid (lemma), dengan memperhatikan konteks kalimat. Berbeda dengan stemming, proses ini idealnya menghasilkan bentuk lema yang valid secara linguistik.

Isi kamus lemma sebaiknya ditentukan dari kata berimbuhan yang benar-benar sering muncul dari data, bukan contoh umum. aturan untuk menentukan niai lema bisasnya menggunakan beberaoa ruls seperti:
1. Kalau imbuhan hanya mengubah bentuk gramatikal tanpa mengubah makna inti (dilantik → lantik), pakai kata dasar.

2. Kalau kata berimbuhan sudah punya makna sendiri yang berbeda dari kata dasarnya (pemerintah, kebijakan, anggaran, alasan), pertahankan bentuk aslinya sebagai lemma.

### Cek

### Code

In [48]:
# Kamus awal; bisa diperluas sesuai domain
lemma_dictionary = {
    "membaca": "baca",
    "membacakan": "baca",
    "dibaca": "baca",
    "bacaan": "baca",
    "berlari": "lari",
    "berjalan": "jalan",
    "berjualan": "jual",
    "penjualan": "jual",
    "membeli": "beli",
    "pembelian": "beli",
    "menulis": "tulis",
    "penulisan": "tulis",
}


In [49]:

def lemmatization(
    df: pl.DataFrame,
    text_col: str,
    output_col: str = "tokens_lemmatized",
    lemma_dict: dict = None
):
    """
    Lemmatization berbasis kamus.
    Kata yang tidak ada dalam kamus dipertahankan.
    """
    if lemma_dict is None:
        lemma_dict = lemma_dictionary

    def lemmatize_text(value):
        if value is None:
            return []
        # kolom list di Polars masuk sebagai pl.Series -> ubah ke list Python
        if isinstance(value, pl.Series):
            tokens = value.to_list()
        elif isinstance(value, list):
            tokens = value
        else:
            tokens = str(value).split()

        return [lemma_dict.get(str(t).lower(), str(t).lower()) for t in tokens]

    return df.with_columns(
        pl.col(text_col).map_elements(
            lemmatize_text,
            return_dtype=pl.List(pl.Utf8)
        ).alias(output_col)
    )

In [50]:
df_lemmatized = lemmatization(
    df_no_stopwords, text_col="tokens_no_stopwords"
)

In [51]:
df_lemmatized

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_lemmatized
str,str,str,list[str],list[str],list[str]
"""Elkan Baggott mengaku bangga j…","""Elkan Baggott mengaku bangga j…","""elkan baggott mengaku bangga j…","[""elkan"", ""baggott"", … ""2026""]","[""elkan"", ""baggott"", … ""2026""]","[""elkan"", ""baggott"", … ""2026""]"
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]","[""ngeliat"", ""banyaknya"", … ""negara""]","[""ngeliat"", ""banyaknya"", … ""negara""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]","[""terpantau"", ""akun"", … ""si""]","[""terpantau"", ""akun"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]"
…,…,…,…,…,…
"""Apa pun Analisisnya, Mari Raya…","""Apa pun Analisisnya, Mari Raya…","""apa pun analisisnya, mari raya…","[""apa"", ""pun"", … ""kemudian""]","[""analisisnya"", ""mari"", … ""lakukan""]","[""analisisnya"", ""mari"", … ""lakukan""]"
"""Pantesan dukun Thailand nya ka…","""Pantesan dukun Thailand nya ka…","""pantesan dukun thailand nya ka…","[""pantesan"", ""dukun"", … ""kalah""]","[""pantesan"", ""dukun"", … ""kalah""]","[""pantesan"", ""dukun"", … ""kalah""]"
"""Setuju sih ini, thailand kalah…","""Setuju sih ini, thailand kalah…","""setuju sih ini, thailand kalah…","[""setuju"", ""sih"", … ""semalem""]","[""setuju"", ""thailand"", … ""semalem""]","[""setuju"", ""thailand"", … ""semalem""]"


## Hasil

In [52]:
columns_to_view = [
    "text",
    "text_clean",
    "text_normalized",
    "tokens",
    "tokens_no_stopwords",
    "tokens_stemmed",
    "tokens_lemmatized"
]

df_comparison = (
    df_lemmatized
    .with_columns(df_stemmed["tokens_stemmed"])
    .select(columns_to_view)
)

df_comparison.head()

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed,tokens_lemmatized
str,str,str,list[str],list[str],list[str],list[str]
"""Elkan Baggott mengaku bangga j…","""Elkan Baggott mengaku bangga j…","""elkan baggott mengaku bangga j…","[""elkan"", ""baggott"", … ""2026""]","[""elkan"", ""baggott"", … ""2026""]","[""elkan"", ""baggott"", … ""2026""]","[""elkan"", ""baggott"", … ""2026""]"
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]","[""balap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]","[""ngeliat"", ""banyaknya"", … ""negara""]","[""ngeliat"", ""banyak"", … ""negara""]","[""ngeliat"", ""banyaknya"", … ""negara""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]","[""terpantau"", ""akun"", … ""si""]","[""pantau"", ""akun"", … ""si""]","[""terpantau"", ""akun"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]"


In [53]:
df_export = df_comparison.clone()

token_columns = [
    "tokens",
    "tokens_no_stopwords",
    "tokens_stemmed",
    "tokens_lemmatized"
]

# Ubah list token menjadi string
for col in token_columns:
    df_export = df_export.with_columns(
        df_export[col].map_elements(
            lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens),
            return_dtype=pl.Utf8
        ).alias(col)
    )

# # Simpan hasil ke CSV
# df_export.write_csv(
#     "hasil_preprocessing.csv",
#     encoding="utf8"   # Polars tidak pakai "utf-8-sig", cukup "utf8"
# )


In [54]:
df_export = df_comparison.with_columns(
    [pl.col(c).list.join(" ") for c in token_columns]
)

df_export.write_csv("hasil_preprocessing2.csv")

In [55]:
df_export.write_csv("hasil_preprocessing2.csv")


In [56]:
df_load = pl.read_csv('hasil_preprocessing2.csv')

In [57]:
df_load

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed,tokens_lemmatized
str,str,str,str,str,str,str
"""Elkan Baggott mengaku bangga j…","""Elkan Baggott mengaku bangga j…","""elkan baggott mengaku bangga j…","""elkan baggott mengaku bangga j…","""elkan baggott mengaku bangga o…","""elkan baggott aku bangga orang…","""elkan baggott mengaku bangga o…"
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","""pembalap motogp fermin aldegue…","""pembalap motogp fermin aldegue…","""balap motogp fermin aldeguer r…","""pembalap motogp fermin aldegue…"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ajaib…","""ngeliat banyaknya reaksi ajaib…","""ngeliat banyak reaksi ajaib or…","""ngeliat banyaknya reaksi ajaib…"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","""terpantau akun tidak reaksi ti…","""pantau akun tidak reaksi timna…","""terpantau akun tidak reaksi ti…"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","""baca di sini calvin verdonk ta…","""baca calvin verdonk tak meraya…","""baca calvin verdonk tak raya h…","""baca calvin verdonk tak meraya…"
…,…,…,…,…,…,…
"""Apa pun Analisisnya, Mari Raya…","""Apa pun Analisisnya, Mari Raya…","""apa pun analisisnya, mari raya…","""apa pun analisisnya mari rayak…","""analisisnya mari rayakan indon…","""analisis mari raya indonesia b…","""analisisnya mari rayakan indon…"
"""Pantesan dukun Thailand nya ka…","""Pantesan dukun Thailand nya ka…","""pantesan dukun thailand nya ka…","""pantesan dukun thailand nya ka…","""pantesan dukun thailand kalah""","""pantesan dukun thailand kalah""","""pantesan dukun thailand kalah"""
"""Setuju sih ini, thailand kalah…","""Setuju sih ini, thailand kalah…","""setuju sih ini, thailand kalah…","""setuju sih ini thailand kalah …","""setuju thailand kalah terhorma…","""tuju thailand kalah hormat sal…","""setuju thailand kalah terhorma…"
